In [ ]:
print("="*70)
print("RESUMEN DEL MODELO")
print("="*70)
print(f"\n📁 Modelo: {MODEL_PATH}")
print(f"✓ Modelo cargado: {os.path.exists(MODEL_PATH)}")
print(f"\n🎯 Configuración:")
print(f"  - Tamaño de entrada: {IMG_WIDTH}x{IMG_HEIGHT}")
print(f"  - Número de clases: {NUM_CLASSES}")
print(f"  - Total de anchors: {ANCHORS.shape[0]}")
print(f"  - Feature maps: {len(feature_map_sizes)}")

# Tamaño del modelo
if os.path.exists(MODEL_PATH):
    model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"  - Tamaño del modelo: {model_size_mb:.2f} MB")

print(f"\n💡 Uso recomendado:")
print(f"  - Umbral de confianza: 0.5 (ajustar según necesidad)")
print(f"  - Umbral NMS: 0.45")
print(f"  - Tipos de entrada: Imágenes RGB")
print(f"  - Clases detectadas: vehículos (cars)")

print(f"\n🚀 Para usar el modelo en producción:")
print(f"  1. Cargar con las clases personalizadas (SSDModel, SSDBoxLoss, SSDClassLoss)")
print(f"  2. Generar los anchors con las mismas configuraciones")
print(f"  3. Preprocesar: resize a {IMG_WIDTH}x{IMG_HEIGHT}, normalizar /255.0")
print(f"  4. Inferencia: model(input_batch, training=False)")
print(f"  5. Postprocesar: decode boxes, aplicar NMS")

print("="*70)

## 13. Resumen del Modelo

In [ ]:
if len(val_images) > 0:
    # Seleccionar una imagen para experimentar
    test_image = val_images[0]
    
    # Cargar imagen una sola vez
    image = cv2.imread(test_image)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, (IMG_WIDTH, IMG_HEIGHT))
    image_normalized = image_resized / 255.0
    image_batch = np.expand_dims(image_normalized, axis=0).astype(np.float32)
    
    # Inferencia una sola vez
    predictions = model(image_batch, training=False)
    box_preds = predictions['boxes']
    class_preds = predictions['classes']
    
    # Probar diferentes umbrales
    confidence_thresholds = [0.3, 0.5, 0.7]
    
    print(f"Probando diferentes umbrales con: {os.path.basename(test_image)}\n")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for idx, conf_th in enumerate(confidence_thresholds):
        # Postprocesar con diferente umbral
        boxes, scores, classes = postprocess_predictions(
            box_preds, class_preds, ANCHORS,
            conf_threshold=conf_th,
            nms_threshold=0.45
        )
        
        boxes_np = boxes.numpy()
        scores_np = scores.numpy()
        classes_np = classes.numpy()
        
        # Visualizar en subplot
        ax = axes[idx]
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        
        img_h, img_w = image.shape[:2]
        num_det = 0
        
        for box, score, cls in zip(boxes_np, scores_np, classes_np):
            if score < conf_th:
                continue
            
            cx, cy, w, h = box
            x1 = int((cx - w / 2) * img_w)
            y1 = int((cy - h / 2) * img_h)
            x2 = int((cx + w / 2) * img_w)
            y2 = int((cy + h / 2) * img_h)
            
            width = x2 - x1
            height = y2 - y1
            
            rect = plt.Rectangle(
                (x1, y1), width, height,
                fill=False, edgecolor='lime', linewidth=2
            )
            ax.add_patch(rect)
            
            label = f"car {score:.2f}"
            ax.text(
                x1, y1 - 5, label,
                color='white', fontsize=8,
                bbox=dict(facecolor='lime', alpha=0.7, edgecolor='none')
            )
            
            num_det += 1
        
        ax.set_title(f"Conf={conf_th} ({num_det} detecciones)", fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## 12. Experimentar con Diferentes Umbrales

Puedes ajustar los umbrales de confianza y NMS para ver cómo afectan las detecciones.

In [ ]:
import time

if len(val_images) > 0:
    # Seleccionar 10 imágenes para benchmark
    num_benchmark = min(10, len(val_images))
    benchmark_images = val_images[:num_benchmark]
    
    print(f"Midiendo rendimiento con {num_benchmark} imágenes...\n")
    
    inference_times = []
    total_detections = []
    
    for img_path in benchmark_images:
        # Cargar y preprocesar
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image_rgb, (IMG_WIDTH, IMG_HEIGHT))
        image_normalized = image_resized / 255.0
        image_batch = np.expand_dims(image_normalized, axis=0).astype(np.float32)
        
        # Medir tiempo de inferencia
        start_time = time.time()
        
        # Inferencia
        predictions = model(image_batch, training=False)
        box_preds = predictions['boxes']
        class_preds = predictions['classes']
        
        # Postprocesar
        boxes, scores, classes = postprocess_predictions(
            box_preds, class_preds, ANCHORS,
            conf_threshold=0.5,
            nms_threshold=0.45
        )
        
        end_time = time.time()
        
        inference_time = (end_time - start_time) * 1000  # en ms
        inference_times.append(inference_time)
        total_detections.append(len(boxes))
    
    # Estadísticas
    avg_time = np.mean(inference_times)
    std_time = np.std(inference_times)
    min_time = np.min(inference_times)
    max_time = np.max(inference_times)
    fps = 1000 / avg_time
    avg_detections = np.mean(total_detections)
    
    print("="*60)
    print("RESULTADOS DEL BENCHMARK")
    print("="*60)
    print(f"Tiempo promedio de inferencia: {avg_time:.2f} ms (±{std_time:.2f} ms)")
    print(f"Tiempo mínimo: {min_time:.2f} ms")
    print(f"Tiempo máximo: {max_time:.2f} ms")
    print(f"FPS estimado: {fps:.2f}")
    print(f"Detecciones promedio por imagen: {avg_detections:.1f}")
    print("="*60)

## 11. Análisis de Rendimiento

Vamos a medir la velocidad de inferencia del modelo.

In [ ]:
# Cambia esta ruta por la imagen que quieras probar
CUSTOM_IMAGE_PATH = "../notebooks/UA-DETRAC-1/valid/images/img00001.jpg"

# Si tienes una imagen propia, colócala en el directorio y cambia la ruta:
# CUSTOM_IMAGE_PATH = "../mi_imagen.jpg"

if os.path.exists(CUSTOM_IMAGE_PATH):
    print(f"Probando con imagen personalizada: {CUSTOM_IMAGE_PATH}\n")
    
    boxes, scores, classes = predict_image(
        CUSTOM_IMAGE_PATH,
        conf_threshold=0.4,  # Puedes ajustar estos umbrales
        nms_threshold=0.45,
        visualize=True
    )
else:
    print(f"⚠️ No se encontró la imagen: {CUSTOM_IMAGE_PATH}")
    print("Cambia CUSTOM_IMAGE_PATH por la ruta de una imagen válida.")

## 10. Probar con Imagen Personalizada

Puedes probar el modelo con cualquier imagen. Solo cambia la ruta en la celda siguiente.

In [ ]:
if len(val_images) > 0:
    # Probar con varias imágenes
    num_test_images = min(5, len(val_images))
    
    print(f"Probando con {num_test_images} imágenes aleatorias...\n")
    
    # Seleccionar imágenes aleatorias
    import random
    test_images = random.sample(val_images, num_test_images)
    
    for i, img_path in enumerate(test_images, 1):
        print(f"\n{'='*60}")
        print(f"Imagen {i}/{num_test_images}: {os.path.basename(img_path)}")
        print('='*60)
        
        boxes, scores, classes = predict_image(
            img_path,
            conf_threshold=0.5,
            nms_threshold=0.45,
            visualize=True
        )

### Probar con múltiples imágenes

In [ ]:
if len(val_images) > 0:
    # Probar con la primera imagen
    test_image = val_images[0]
    print(f"Probando con: {os.path.basename(test_image)}\n")
    
    boxes, scores, classes = predict_image(
        test_image,
        conf_threshold=0.5,
        nms_threshold=0.45,
        visualize=True
    )

### Probar con la primera imagen

In [ ]:
# Obtener imágenes de validación
val_images = glob.glob(os.path.join(VAL_IMAGES_DIR, "*.jpg"))

if len(val_images) == 0:
    print(f"⚠️ No se encontraron imágenes en {VAL_IMAGES_DIR}")
    print("Verifica que la ruta del dataset sea correcta.")
else:
    print(f"✓ Encontradas {len(val_images)} imágenes de validación")
    print(f"\nPrimeras 5 imágenes:")
    for i, img in enumerate(val_images[:5], 1):
        print(f"  {i}. {os.path.basename(img)}")

## 9. Probar con Imágenes de Validación

Vamos a probar el modelo con algunas imágenes del conjunto de validación.

In [ ]:
def predict_image(image_path, conf_threshold=0.5, nms_threshold=0.45, visualize=True):
    """
    Hacer predicción en una imagen.
    
    Args:
        image_path: Ruta a la imagen
        conf_threshold: Umbral de confianza
        nms_threshold: Umbral para NMS
        visualize: Si True, muestra la imagen con predicciones
    
    Returns:
        boxes, scores, classes
    """
    # Cargar imagen
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"No se pudo cargar la imagen: {image_path}")
    
    original_h, original_w = image.shape[:2]
    
    # Preprocesar
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, (IMG_WIDTH, IMG_HEIGHT))
    image_normalized = image_resized / 255.0
    image_batch = np.expand_dims(image_normalized, axis=0).astype(np.float32)
    
    # Inferencia
    predictions = model(image_batch, training=False)
    box_preds = predictions['boxes']
    class_preds = predictions['classes']
    
    # Postprocesar
    boxes, scores, classes = postprocess_predictions(
        box_preds, class_preds, ANCHORS,
        conf_threshold=conf_threshold,
        nms_threshold=nms_threshold
    )
    
    # Convertir a numpy
    boxes_np = boxes.numpy()
    scores_np = scores.numpy()
    classes_np = classes.numpy()
    
    # Visualizar
    if visualize:
        num_det = visualize_predictions(
            image, boxes_np, scores_np, classes_np,
            conf_threshold=conf_threshold,
            title=f"Imagen: {os.path.basename(image_path)}"
        )
        print(f"Detecciones encontradas: {num_det}")
        print(f"Tamaño original: {original_w}x{original_h}")
    
    return boxes_np, scores_np, classes_np

print("✓ Función de predicción definida")

## 8. Función para Hacer Predicciones

In [ ]:
def visualize_predictions(image, boxes, scores, classes, conf_threshold=0.5, title="Predicciones"):
    """
    Visualizar predicciones en una imagen.
    
    Args:
        image: Imagen BGR (OpenCV format)
        boxes: Cajas delimitadoras normalizadas (cx, cy, w, h)
        scores: Puntuaciones de confianza
        classes: Clases predichas
        conf_threshold: Umbral de confianza para mostrar
        title: Título de la figura
    """
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Convertir BGR a RGB para matplotlib
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(image_rgb)
    
    img_h, img_w = image.shape[:2]
    
    # Dibujar cada detección
    num_detections = 0
    for box, score, cls in zip(boxes, scores, classes):
        if score < conf_threshold:
            continue
        
        # Convertir de (cx, cy, w, h) normalizado a coordenadas de píxeles
        cx, cy, w, h = box
        x1 = int((cx - w / 2) * img_w)
        y1 = int((cy - h / 2) * img_h)
        x2 = int((cx + w / 2) * img_w)
        y2 = int((cy + h / 2) * img_h)
        
        width = x2 - x1
        height = y2 - y1
        
        # Dibujar rectángulo
        rect = plt.Rectangle(
            (x1, y1), width, height,
            fill=False, edgecolor='lime', linewidth=2
        )
        ax.add_patch(rect)
        
        # Etiqueta con confianza
        label = f"car {score:.2f}"
        ax.text(
            x1, y1 - 5,
            label,
            color='white',
            fontsize=10,
            bbox=dict(facecolor='lime', alpha=0.7, edgecolor='none')
        )
        
        num_detections += 1
    
    ax.set_title(f"{title} - {num_detections} detecciones", fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return num_detections

print("✓ Función de visualización definida")

## 7. Función para Visualizar Predicciones

In [ ]:
@tf.function
def decode_boxes(encoded_boxes, anchors):
    """Decode box predictions relative to anchor boxes."""
    cx_a, cy_a, w_a, h_a = tf.split(anchors, 4, axis=-1)
    dx, dy, dw, dh = tf.split(encoded_boxes, 4, axis=-1)
    
    cx = cx_a + dx * w_a
    cy = cy_a + dy * h_a
    w = w_a * tf.exp(dw)
    h = h_a * tf.exp(dh)
    
    return tf.concat([cx, cy, w, h], axis=-1)


def non_maximum_suppression(boxes, scores, iou_threshold=0.45):
    """Apply NMS to filter overlapping boxes."""
    cx, cy, w, h = tf.split(boxes, 4, axis=-1)
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    
    corners = tf.concat([y1, x1, y2, x2], axis=-1)
    corners = tf.squeeze(corners, axis=0)
    scores_1d = tf.squeeze(scores, axis=[0, 2])
    
    selected_indices = tf.image.non_max_suppression(
        corners, scores_1d,
        max_output_size=100,
        iou_threshold=iou_threshold
    )
    
    selected_boxes = tf.gather(boxes[0], selected_indices)
    selected_scores = tf.gather(scores[0], selected_indices)
    
    return selected_boxes, selected_scores


def postprocess_predictions(box_preds, class_preds, anchors, conf_threshold=0.5, nms_threshold=0.45):
    """Full postprocessing pipeline."""
    # Decodificar boxes
    decoded_boxes = decode_boxes(box_preds, anchors)
    
    # Obtener probabilidades de clase
    class_scores = tf.nn.sigmoid(class_preds)
    
    # Filtrar por umbral de confianza
    mask = class_scores[0, :, 0] >= conf_threshold
    filtered_boxes = tf.expand_dims(tf.boolean_mask(decoded_boxes[0], mask), axis=0)
    filtered_scores = tf.expand_dims(tf.boolean_mask(class_scores[0], mask), axis=0)
    
    # Si no hay detecciones, retornar vacío
    if tf.shape(filtered_boxes)[1] == 0:
        return (
            tf.zeros((0, 4), dtype=tf.float32),
            tf.zeros((0,), dtype=tf.float32),
            tf.zeros((0,), dtype=tf.int32),
        )
    
    # Aplicar NMS
    nms_boxes, nms_scores = non_maximum_suppression(
        filtered_boxes, filtered_scores, nms_threshold
    )
    
    # Clases (todas son 0 para single-class)
    nms_classes = tf.zeros((tf.shape(nms_boxes)[0],), dtype=tf.int32)
    
    return nms_boxes, nms_scores[:, 0], nms_classes

print("✓ Funciones de postprocesamiento definidas")

## 6. Funciones de Postprocesamiento

Funciones para decodificar y filtrar las predicciones del modelo.

In [ ]:
def generate_anchors_for_feature_map(feature_map_size, img_width, img_height, scales, aspect_ratios):
    """Generate anchor boxes for one feature map."""
    fm_h, fm_w = feature_map_size
    anchors = []
    
    for i in range(fm_h):
        for j in range(fm_w):
            cx = (j + 0.5) / fm_w
            cy = (i + 0.5) / fm_h
            
            for scale in scales:
                for ar in aspect_ratios:
                    w = scale * np.sqrt(ar)
                    h = scale / np.sqrt(ar)
                    anchors.append([cx, cy, w, h])
    
    return np.array(anchors, dtype=np.float32)


def generate_all_anchors(feature_map_sizes, img_width, img_height):
    """Generate all anchors for multiple feature maps."""
    all_anchors = []
    
    # Escalas para cada feature map
    scales_list = [
        [0.1, 0.15],   # Feature map 1 (más grande, objetos pequeños)
        [0.2, 0.3],    # Feature map 2 (medio)
        [0.4, 0.5],    # Feature map 3 (más pequeño, objetos grandes)
    ]
    
    # Aspect ratios comunes
    aspect_ratios = [0.5, 1.0, 1.5, 2.0]
    
    for fm_size, scales in zip(feature_map_sizes, scales_list):
        anchors = generate_anchors_for_feature_map(
            fm_size, img_width, img_height, scales, aspect_ratios
        )
        all_anchors.append(anchors)
    
    return tf.constant(np.vstack(all_anchors), dtype=tf.float32)


# Generar anchors para las feature maps del modelo
# Feature map sizes para input de 180x320
feature_map_sizes = [
    (23, 40),  # ~180/8 x 320/8
    (12, 20),  # ~180/16 x 320/16
    (6, 10),   # ~180/32 x 320/32
]

ANCHORS = generate_all_anchors(feature_map_sizes, IMG_WIDTH, IMG_HEIGHT)

print(f"✓ Anchors generados: {ANCHORS.shape}")
print(f"  - Total de anchors: {ANCHORS.shape[0]}")

## 5. Generar Anchors

Los anchors son necesarios para decodificar las predicciones del modelo.

In [ ]:
print("Cargando modelo...")

try:
    model = keras.models.load_model(
        MODEL_PATH,
        custom_objects={
            'SSDModel': SSDModel,
            'SSDBoxLoss': SSDBoxLoss,
            'SSDClassLoss': SSDClassLoss,
        }
    )
    print("✓ Modelo cargado correctamente")
    print(f"\nResumen del modelo:")
    print(f"  - Inputs: {model.input_shape}")
    print(f"  - Outputs: {[out.shape for out in model.outputs] if isinstance(model.output, list) else model.output_shape}")
    
except Exception as e:
    print(f"❌ Error al cargar el modelo: {e}")
    raise

## 4. Cargar el Modelo Entrenado

In [ ]:
class SSDBoxLoss(keras.losses.Loss):
    """Smooth L1 loss for bounding box regression."""
    
    def __init__(self, name="ssd_box_loss"):
        super().__init__(name=name)
    
    def call(self, y_true, y_pred):
        diff = y_pred - y_true
        abs_diff = tf.abs(diff)
        smooth_l1 = tf.where(abs_diff < 1.0, 0.5 * diff**2, abs_diff - 0.5)
        return tf.reduce_sum(smooth_l1, axis=-1)


class SSDClassLoss(keras.losses.Loss):
    """Binary cross-entropy for single-class detection."""
    
    def __init__(self, name="ssd_class_loss"):
        super().__init__(name=name)
    
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true, logits=y_pred)
        return tf.reduce_sum(bce, axis=-1)


class SSDModel(keras.Model):
    """SSD model wrapper."""
    
    def __init__(self, base_model, box_loss, class_loss, name="ssd_model"):
        super().__init__(name=name)
        self.base_model = base_model
        self.box_loss_fn = box_loss
        self.class_loss_fn = class_loss
    
    def call(self, inputs, training=None):
        return self.base_model(inputs, training=training)
    
    def get_config(self):
        return {"name": self.name}
    
    @classmethod
    def from_config(cls, config, custom_objects=None):
        return cls(None, None, None, name=config.get("name", "ssd_model"))

print("✓ Clases personalizadas definidas")

## 3. Definir Clases Personalizadas

Estas clases son necesarias para cargar el modelo correctamente.

In [ ]:
# Configurar rutas
DESTINO = os.getcwd()
os.chdir(DESTINO)

# Rutas del modelo
MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, "ssd_mobilenet_car_detector.keras")

# Rutas del dataset (para imágenes de prueba)
DATASET_DIR = "../notebooks/UA-DETRAC-1"
VAL_IMAGES_DIR = os.path.join(DATASET_DIR, "valid/images")

# Parámetros del modelo
IMG_WIDTH = 320
IMG_HEIGHT = 180
NUM_CLASSES = 1

print(f"Directorio actual: {DESTINO}")
print(f"Modelo: {MODEL_PATH}")
print(f"Existe modelo: {os.path.exists(MODEL_PATH)}")

## 2. Configuración de Rutas

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import glob

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")

## 1. Importar Librerías

# Test del Modelo SSD - Detección de Vehículos

Este notebook permite probar el modelo SSD entrenado para detección de vehículos.

**Características:**
- Carga el modelo guardado
- Prueba con imágenes de validación
- Visualiza las predicciones
- Prueba con imágenes personalizadas